# Futures Market Making

In this exercise, you will examine the exchange trade data to
approximate the market making opportunities, corresponding
risk, and consider how the approximation could be improved.

## 1. Edge given up by market takers

### 1a. November

Use CME trade data from Databento to approximate the P&L
available to market makers in November crude oil on
2026-09-24. That is, assume the trades with a buy or sell
aggressor are trading against a limit order from a
market maker. Consider the time period from 8:30 AM Chicago
until settlement. Assume the market makers start flat
at 8:30 AM to keep the approximation limited to this
time range.

You should compile a data set of trades with a time, side,
price, and quantity.
From that, determine total
P&L for the day, the largest open position intraday, and
the final open position at the end of the day.

Then decompose the total P&L for the day into
- edge: P&L generated by trading better than a theoretical
  price. In this case, use the mid-price from just before the trade.
- inventory: P&L generated by holding a position as the price moves.
Note that if one component of the decomposition is
computed, then the other can be computed as the difference from
the total.

### 1b. December

Redo 1a. for the December contract.

### 1c. November-December Spread

Redo 1a. for the November-December spread contract.

## 1a. November contract

In [ ]:
import datetime
from zoneinfo import ZoneInfo

import databento as db
import numpy as np
import pandas as pd

from finm37000 import (
    disk_cache,
    get_databento_api_key,
    get_stats_by_date,
    temp_env,
)

with temp_env(DATABENTO_API_KEY=get_databento_api_key()):
    client = db.Historical()

dataset = "GLBX.MDP3"
symbol = "CLX6"
trade_date = datetime.date(2026, 9, 24)

tz_chicago = ZoneInfo("America/Chicago")
start = pd.Timestamp("2026-09-24 08:30:00", tz=tz_chicago)
end = pd.Timestamp("2026-09-24 13:30:00", tz=tz_chicago)


In [13]:
@disk_cache(scope="homework_1a")
def cached_tbbo(dataset, symbol, start, end):
    return client.timeseries.get_range(
        dataset=dataset,
        schema="tbbo",
        symbols=[symbol],
        stype_in="raw_symbol",
        start=start,
        end=end,
    ).to_df()

def get_summary(dataset, symbol, trade_date, start, end,
                settlement_price=None,):
    
    multiplier = 1_000  # Barrels per CL contract

    raw = cached_tbbo(dataset, symbol, start, end)

    # Preserve feed order, including records with identical timestamps.
    trades = raw.reset_index(drop=True).copy()
    trades["time"] = pd.to_datetime(
        trades["ts_event"], utc=True
    ).dt.tz_convert(tz_chicago)

    trades = trades.loc[
        (trades["time"] >= start)
        & (trades["time"] < end)
    ].copy()

    # Inspect how much activity is excluded because its aggressor is unknown.
    display(
        trades.groupby("side")
        .agg(records=("size", "size"), contracts=("size", "sum"))
    )

    # The assignment includes only trades with buy or sell aggressors.
    trades = trades.loc[
        trades["side"].isin(["A", "B"]),
        ["time", "side", "price", "size", "bid_px_00", "ask_px_00"],
    ].rename(columns={"size": "quantity"})

    trades = trades.reset_index(drop=True)

    if trades.empty:
        raise ValueError("No eligible trades found.")

    trades["mid"] = (
        trades["bid_px_00"] + trades["ask_px_00"]
    ) / 2

    valid_quotes = (
        np.isfinite(trades[["bid_px_00", "ask_px_00"]]).all(axis=1)
        & (trades["bid_px_00"] <= trades["ask_px_00"])
    )

    if not valid_quotes.all():
        raise ValueError(
            f"{(~valid_quotes).sum()} trades have missing or crossed quotes; "
            "inspect these before calculating edge."
        )

    display(trades.head())

    

    if settlement_price is None:
        definition_start = pd.Timestamp(trade_date, tz=tz_chicago)
        definition_end = definition_start + pd.DateOffset(days=1)
        definitions = client.timeseries.get_range(
            dataset=dataset,
            schema="definition",
            symbols=[symbol],
            stype_in="raw_symbol",
            start=definition_start,
            end=definition_end,
        ).to_df()
    
        daily_stats = get_stats_by_date(
            client=client,
            date=trade_date,
            defs=definitions,
        )
    
        stats_table = daily_stats.reset_index()
        settlement_rows = stats_table.loc[
            (stats_table["Symbol"] == symbol)
            & (
                pd.to_datetime(stats_table["Trade date"]).dt.date
                == trade_date
            ),
            "Settlement price",
        ].dropna()

        if len(settlement_rows) != 1:
            display(stats_table)
            raise ValueError(f"Expected one official settlement for {symbol} on {trade_date}.")

        settlement_price = float(settlement_rows.iloc[0])

    # Positive = market maker buys; negative = market maker sells.
    trades["mm_quantity"] = (
        trades["side"].map({"A": 1, "B": -1})
        * trades["quantity"].astype("int64")
    )

    trades["position"] = trades["mm_quantity"].cumsum()

    # Accounting term for executions; not actual futures notional cash transfers.
    trades["execution_pnl_term"] = (
        -multiplier * trades["mm_quantity"] * trades["price"]
    )

    trades["edge"] = (
        multiplier
        * trades["mm_quantity"]
        * (trades["mid"] - trades["price"])
    )

    final_position = int(trades["position"].iloc[-1])

    total_pnl = (
        trades["execution_pnl_term"].sum()
        + multiplier * final_position * settlement_price
    )

    edge_pnl = trades["edge"].sum()
    inventory_pnl = total_pnl - edge_pnl

    peak_row = trades.loc[trades["position"].abs().idxmax()]

    summary = pd.Series({
        "Settlement price ($/barrel)": settlement_price,
        "Eligible traded volume (contracts)": trades["quantity"].sum(),
        "Total P&L ($)": total_pnl,
        "Edge P&L ($)": edge_pnl,
        "Inventory P&L ($)": inventory_pnl,
        "Largest absolute position (contracts)": abs(peak_row["position"]),
        "Signed position at that peak (contracts)": peak_row["position"],
        "Time of peak position (Chicago)": peak_row["time"],
        "Final position (contracts)": final_position,
    })

    position_before = trades["position"].shift(1, fill_value=0)
    mid_change = trades["mid"].diff().fillna(0)

    inventory_check = multiplier * (
        (position_before * mid_change).sum()
        + final_position * (settlement_price - trades["mid"].iloc[-1])
    )

    assert np.isclose(inventory_pnl, inventory_check, atol=1e-6)                                                                
    display(summary)
    return summary, trades

In [16]:
dataset = "GLBX.MDP3"
symbol = "CLX6"
trade_date = datetime.date(2026, 9, 24)

tz_chicago = ZoneInfo("America/Chicago")
start = pd.Timestamp("2026-09-24 08:30:00", tz=tz_chicago)
end = pd.Timestamp("2026-09-24 13:30:00", tz=tz_chicago)
nov_summary, nov_trades = get_summary(dataset, symbol, trade_date, start, end)

,records,contracts
side,,
A,37808,74202
B,38612,73976
N,9147,12983


,time,side,price,quantity,bid_px_00,ask_px_00,mid
0,2026-09-24 08:30:00.001223637-05:00,B,93.95,4,93.94,93.95,93.945
1,2026-09-24 08:30:00.002394091-05:00,A,93.94,2,93.94,93.96,93.950
2,2026-09-24 08:30:00.043655889-05:00,A,93.95,2,93.95,93.96,93.955
3,2026-09-24 08:30:00.047908773-05:00,A,93.95,3,93.95,93.96,93.955
4,2026-09-24 08:30:00.064976979-05:00,B,93.95,1,93.94,93.95,93.945


/tmp/ipykernel_1434/1756781214.py:69: BentoWarning: The request time range does not start at UTC midnight, instrument definitions effective on this date may be missing. See: https://databento.com/docs/schemas-and-data-formats/instrument-definitions#snapshots
  definitions = client.timeseries.get_range(


Settlement price ($/barrel)                                               94.61
Eligible traded volume (contracts)                                       148178
Total P&L ($)                                                            7790.0
Edge P&L ($)                                                          1263245.0
Inventory P&L ($)                                                    -1255455.0
Largest absolute position (contracts)                                      1886
Signed position at that peak (contracts)                                  -1886
Time of peak position (Chicago)             2026-09-24 11:04:17.643924263-05:00
Final position (contracts)                                                  226
dtype: object

## 1b december contract

In [18]:
dataset = "GLBX.MDP3"
symbol = "CLZ6"
trade_date = datetime.date(2026, 9, 24)

tz_chicago = ZoneInfo("America/Chicago")
start = pd.Timestamp("2026-09-24 08:30:00", tz=tz_chicago)
end = pd.Timestamp("2026-09-24 13:30:00", tz=tz_chicago)
dec_summary, dec_trades = get_summary(dataset, symbol, trade_date, start, end)

,records,contracts
side,,
A,8733,12940
B,8850,13078
N,6561,9577


,time,side,price,quantity,bid_px_00,ask_px_00,mid
0,2026-09-24 08:30:00.001223459-05:00,B,90.79,4,90.78,90.79,90.785
1,2026-09-24 08:30:00.014769965-05:00,A,90.78,1,90.78,90.80,90.790
2,2026-09-24 08:30:00.427772843-05:00,A,90.80,1,90.80,90.81,90.805
3,2026-09-24 08:30:00.963149235-05:00,B,90.81,1,90.80,90.81,90.805
4,2026-09-24 08:30:01.001367429-05:00,A,90.84,1,90.84,90.85,90.845


/tmp/ipykernel_1434/1756781214.py:69: BentoWarning: The request time range does not start at UTC midnight, instrument definitions effective on this date may be missing. See: https://databento.com/docs/schemas-and-data-formats/instrument-definitions#snapshots
  definitions = client.timeseries.get_range(


Settlement price ($/barrel)                                                90.9
Eligible traded volume (contracts)                                        26018
Total P&L ($)                                                         -154520.0
Edge P&L ($)                                                           193155.0
Inventory P&L ($)                                                     -347675.0
Largest absolute position (contracts)                                       343
Signed position at that peak (contracts)                                    343
Time of peak position (Chicago)             2026-09-24 11:15:44.029974181-05:00
Final position (contracts)                                                 -138
dtype: object

## 1c spread

In [19]:
spread_settlement = (
    nov_summary["Settlement price ($/barrel)"]
    - dec_summary["Settlement price ($/barrel)"]
)

spread_summary, spread_trades = get_summary(
    dataset,
    "CLX6-CLZ6",
    trade_date,
    start,
    end,
    settlement_price=spread_settlement,
)

,records,contracts
side,,
A,7353,20958
B,5848,21917
N,6297,11260


,time,side,price,quantity,bid_px_00,ask_px_00,mid
0,2026-09-24 08:30:00.013546383-05:00,A,3.16,1,3.16,3.17,3.165
1,2026-09-24 08:30:00.160555835-05:00,B,3.16,1,3.15,3.16,3.155
2,2026-09-24 08:30:01.525767329-05:00,B,3.17,6,3.16,3.17,3.165
3,2026-09-24 08:30:26.047331885-05:00,B,3.17,1,3.16,3.17,3.165
4,2026-09-24 08:30:26.053519989-05:00,B,3.17,1,3.16,3.17,3.165


Settlement price ($/barrel)                                                3.71
Eligible traded volume (contracts)                                        42875
Total P&L ($)                                                         -667690.0
Edge P&L ($)                                                           272975.0
Inventory P&L ($)                                                     -940665.0
Largest absolute position (contracts)                                      2168
Signed position at that peak (contracts)                                  -2168
Time of peak position (Chicago)             2026-09-24 12:38:42.831623301-05:00
Final position (contracts)                                                 -959
dtype: object

## 2. Margining

How much margin is required to maintain the end-of-day positions in problem 1.
Answer this for each contract individually, then as a combined position.
Exact requirements can be found here
https://www.cmegroup.com/markets/energy/crude-oil/light-sweet-crude.margins.html#marginsTab=OUTRIGHT&pageNumber=1
but for simplicity, let's
call it $\$9000$ for November (long or short) per contract,
$\$8000$ for December, and $\$2000$ for the Nov-Dec spread.

In [20]:
q_nov = int(nov_summary["Final position (contracts)"])
q_dec = int(dec_summary["Final position (contracts)"])
q_spread = int(spread_summary["Final position (contracts)"])

# Each position margined separately.
nov_margin = 9_000 * abs(q_nov)
dec_margin = 8_000 * abs(q_dec)
spread_margin = 2_000 * abs(q_spread)

separate_margin = nov_margin + dec_margin + spread_margin

# Net exposures after converting the spread into its legs.
net_nov = q_nov + q_spread
net_dec = q_dec - q_spread

spread_pairs = (
    min(abs(net_nov), abs(net_dec))
    if net_nov * net_dec < 0
    else 0
)

remaining_nov = abs(net_nov) - spread_pairs
remaining_dec = abs(net_dec) - spread_pairs

combined_margin = (
    2_000 * spread_pairs
    + 9_000 * remaining_nov
    + 8_000 * remaining_dec
)

margin_summary = pd.Series({
    "November margin separately ($)": nov_margin,
    "December margin separately ($)": dec_margin,
    "Nov-Dec spread margin separately ($)": spread_margin,
    "Total margin separately ($)": separate_margin,
    "Combined net November position": net_nov,
    "Combined net December position": net_dec,
    "Combined spread pairs": spread_pairs,
    "Combined margin ($)": combined_margin,
    "Margin reduction from combining ($)": (
        separate_margin - combined_margin
    ),
})

display(margin_summary)

November margin separately ($)          2034000
December margin separately ($)          1104000
Nov-Dec spread margin separately ($)    1918000
Total margin separately ($)             5056000
Combined net November position             -733
Combined net December position              821
Combined spread pairs                       733
Combined margin ($)                     2170000
Margin reduction from combining ($)     2886000
dtype: int64

## 3. Estimating Opportunity

Your boss says, "Let's join the other market makers,
quoting 10\% of their quantities, then we'll capture 1/11 of the
market making P&L in question 1". Critique this assessment.
Address the ways in which the calculations in 1 are
both over and under estimates.
Then provide a brief proposal for how you could provide
a more accurate simulation to better estimate
the opportunity in this market.

I think there are a few issues with this assessment.  We might get selection bias on which orders we trade, and we might under-estimate the losses.  We also over-estimate the edge because we assume the mid market is fair.  I think the best way to assess the opportunity in the market is to simulate how we would place our orders, and make some assumptions of when our limit orders will get filled, how often we reload, and how we adjust our quoting to adjust for the risk we take on.  